# On Qwen, can the pass test fail a split that drops the fine-tuning update?

*E44, `experiments/qwen_maps/`, run 2026-10-28, written 2026-10-29. Plan: `reports/plans/2026-10-28_plan_qwen-q0q1.md`.
Diagnosis: `reports/analyses/2026-10-29_analysis_e44-no-teeth.md`.*

## The answer

No. The unchanged decomposition read the same as in the earlier run, and the split pieces were computed precisely, so
the apparatus works. But all three deliberately bad placements of the update passed the test's key reading. The
reason is in the design, not a bug. The importance function computes masks from each state's own activations, and on
the fine-tuned weights those masks carry the update by themselves. A state that leaves the whole update out of its
subcomponents still keeps 97% of it. As planned, the Qwen line stops here until the user decides what the pass test
should measure.

## What we were trying to test

`GOALS.md` asks whether a post-training update can be written into a parameter decomposition so that the decomposition
stays faithful. On Qwen3-1.7B we have a VPD decomposition of 196 weight matrices. Each is written as $W = U V^\top + \Delta$,
where the columns of $U$ and $V$ are the subcomponents and $\Delta$ is the part they do not explain. The
importance function Γ reads the model's activations and gives each subcomponent a causal importance between 0 and 1.
A CI-masked model multiplies each subcomponent by its importance and leaves $\Delta$ out.

The update is E40's fine-tune of Qwen on maths answers, saved after 1 and 10 Adam steps. A *state* is a decomposition
that holds the update somewhere: in changed subcomponents, in $\Delta$, or partly in each. Before trusting any test of
the real splits, the plan asked the test to fail three bad states:

- **Do-nothing** puts the whole update in $\Delta$ and leaves the subcomponents alone. Since the masked model drops
  $\Delta$, it should lose the update. This rules out a test that passes anything holding the fine-tuned weights.
- **Two halves.** The least-squares split restricted to the half of each matrix's subcomponents that Γ rarely
  switches on for maths text, against the half it often switches on. Writing the update where Γ looks away should
  hurt more. This rules out a test blind to where the update is placed.
- **Random.** A random change to the subcomponents, scaled so that each matrix's output on maths inputs moves as much
  as under the update. It should push the readings over the line, showing the test notices a different model.

The test's key reading is the *share kept*, $r_2 = (K(D) - K)/(K(D) - K_0)$, read on 128 maths rows of 512 tokens.
Here $K$ is the mean KL per token from the fine-tuned model to the state's CI-masked model. The reference $K(D)$ is
the same reading for the unchanged decomposition, which ignores the update. The floor $K_0$ is the unchanged
decomposition read against the original model. So $r_2$ reads 0 for a state that drops the update. It reads 1 for one
that matches the fine-tuned model as well as the decomposition matches the original. A state passes with
$r_2 \ge 0.8$ and two ratios to the unchanged decomposition at most 1.5. One is its masked KL against its own
unmasked model; the other is its KL under saved adversarial masks.

The real splits were to follow: least squares, least squares without the part outside the subcomponents' span, and
the scaled adjoint split. They were built but not read, because the controls came first.

## What we expected, and why

The plan filed five predictions. First, the unchanged decomposition would read within 1% of E40's values, since the
same code on the same rows had matched to 0.09% before. Second, the rarely-used half would keep at least 0.2 less of
the update than the often-used half, provided both halves left similar shares of the update in $\Delta$. Third, the
random change would push at least one ratio over 1.5. Fourth, do-nothing would keep less than 0.8 of the update; the
plan expected about 0, because the masked model runs without $\Delta$. Fifth, if any of the three controls failed to
fail at step 10, the line would stop and the pass criterion would go back to the user.

## What we observed

### Does the unchanged decomposition read as it did in E40?

![W0 readings against E40's values](../experiments/qwen_maps/figures/reproduction.png)

**Reading it.** Yes: every reading of the unchanged decomposition matched E40 to within 1%. The figure plots each
reading as a percent difference from E40's value, with the filed tolerance as the grey band. The masked KL on both
row sets and all four replayed adversaries sit on the line. The KL from the original to the fine-tuned model at steps
1, 10 and 100 also sits inside, the largest gap being 0.6% at step 100 on the scored pod. The first pod, run on
weights saved before they were regenerated, agrees as closely.

### Are the split pieces computed precisely?

![stored precision error per matrix](../experiments/qwen_maps/figures/precision.png)

**Reading it.** Yes: storing the pieces costs at most two parts in ten thousand of the update, five times under the
stop line. Each point is one matrix: the size of the change the stored pieces apply, minus the float64 change, over
the update's size. The least-squares split's worst matrix is below one part in a million. The rarely-used half's tail
reaches 0.00016 at step 10, in the up matrices of the first few MLP layers, whose pieces are the worst conditioned. A first pod that stored the pieces in
bf16 missed this line by four times and stopped, so fp32 storage was what made the splits usable.

### Did the two halves move the same amount of the update?

![share of the update left in Delta per state](../experiments/qwen_maps/figures/shares.png)

**Reading it.** No, so the halves comparison was not scored. The figure plots the share of the update each state
leaves in $\Delta$, weighted by the maths inputs; do-nothing leaves all of it. The rarely-used half leaves 0.71 and
the often-used half 0.52, a gap of 0.19 against the filed limit of 0.1. Any difference in what they keep would mix
placement with how much each moved. The full least-squares split leaves about a third of
the update in $\Delta$. That part lies outside the span of the subcomponents, almost all of it in attention matrices.

### Does a state that leaves the update out of its subcomponents lose the update?

![KL to the fine-tuned model per control](../experiments/qwen_maps/figures/kept.png)

**Reading it.** No: do-nothing loses almost none of the update, and the halves lose none either. The figure plots
each control's KL from the fine-tuned model to its masked model. The red dashed line is the unchanged decomposition,
which drops the update, and the dotted line is $K_0$. At step 10 do-nothing reads 0.09 nats against the unchanged
decomposition's 1.42, so it keeps 0.975 of the update where about 0 was expected. Step 1 tells the same story at a
fifth of the scale. Only the random change lands on the red line, as a state that ignores the update should.

The diagnosis found why by reading the code. Do-nothing's subcomponents are identical to the unchanged
decomposition's, and with $\Delta$ off the masked forward never reads the swapped-in weights. The one thing that
differs is the masks: Γ computes them from the fine-tuned model's activations for do-nothing, and from the original
model's for $K(D)$. Those masks alone close 1.32 of the 1.36 nats. Any state that holds the fine-tuned weights
inherits them, which is why both halves read like do-nothing.

### Does a random change of the update's size push any ratio over the line?

![ratios to W0 at step 10](../experiments/qwen_maps/figures/ratios.png)

**Reading it.** No: the random change reads at most 1.0004 times the unchanged decomposition, far under the 1.5
line. The figure plots the four ratios at step 10 for each control. The random change leaves each masked model as
close to its own unmasked model as before. Do-nothing and both halves cross the line only on the maths masked KL, at
about 1.58. Their adversarial ratio on maths text falls to 0.6, which the plan did not expect. One caveat: the random
change was matched matrix by matrix on the size of its change, not on how far it moves the model's output. Its
masked model sits 1.42 nats from the fine-tuned model, the same as the unchanged decomposition, so it barely moved the
output. A random change matched on KL from the original model would be a stronger control.

## What we learned

On Qwen, the pass test as built cannot fail a state that holds the fine-tuned weights. The masks Γ computes from
those weights' activations bring the masked model to the fine-tuned one whatever the subcomponents hold. So the test
measures how well Γ tracks the model it is run on, not whether the update was written into the subcomponents. For
`GOALS.md` this means no split of E40's update was scored, and the weak version is neither supported nor refuted on
Qwen yet. Whether Γ's new masks describe the update or merely copy the fine-tuned model's output is the open question
this raises.

## How compelling is the evidence

The failure is solid. The code leaves no other path from the update into the masked model, and the arithmetic of the
share kept checks to four places at both steps. The reproduction check rules out drift in the apparatus. It is one
update, a large format change, on one set of 128 maths rows; but the mechanism does not depend on the update. The
diagnosis is by elimination from the code. A short cluster job that swaps Γ's inputs between the two models would
confirm it directly; it has not been run.

## Surprises

Something unexpected happened. The plan predicted do-nothing would keep about none of the update, and it kept 97% of
it at both steps. Its masked KL on maths rows was predicted at 8 to 11 times the unchanged decomposition's and read
1.58. The random change, predicted to cross 1.5, never left 1.0. The adversarial ratio on maths rows fell to 0.6 for
every state holding the fine-tuned weights.

## What next

- **The user decides what the pass test measures.** One option: Γ reads the original model's activations for every
  state, as $K(D)$ does. Then do-nothing keeps 0 by definition and the controls can be rerun. The other option reads
  something other than closeness to the fine-tuned model. Either changes the pass criterion fixed on 2026-10-19. The
  Qwen line and E45's reading job wait on the choice.
- **Fable's advice** is in `reports/plans/2026-10-29_advice_e44-pass-test-FABLE.md`. It recommends the first option,
  rerun with a random control matched on KL from the original model. Before the card, it proposes one $0 cluster
  job: feed Γ the fine-tuned model's residual stream up to one layer and the original's after it. If the masks need
  the last layers to reproduce the update, Γ is copying the output; if early layers suffice, it reads the change.
## Scorecard

| id | claim in words | rule | read | verdict |
|---|---|---|---|---|
| control_W0 | the unchanged decomposition reads as in E40 | every reading within 1% | largest gap 0.63% | holds |
| teeth_i | the rarely-used half keeps less than the often-used half | shares within 0.1, then $r_2$ lower ≤ upper − 0.2 | shares 0.71 and 0.52 | fails: confounded by the span, steps 1 and 10 |
| teeth_ii | a random change crosses a ratio of 1.5 | largest ratio > 1.5 | 1.0002 and 1.0004 | fails, steps 1 and 10 |
| teeth_iii | do-nothing keeps less than 0.8 of the update | $r_2$ < 0.8 | 0.977 and 0.975 | fails, steps 1 and 10 |
| teeth_iii, web | do-nothing's adversarial ratio on web rows stays under 1.5 at step 10 | ratio ≤ 1.5 | 1.02 | holds |
| teeth outcome | all three controls fail to pass at step 10 | all of the above | none did | no teeth: line stops before the splits |

## Not run

- Stage M: the least-squares split, the in-span least-squares state and the scaled adjoint split were built in
  stage S but never read; the line stopped first.
- Stage G: maths accuracy of the fine-tuned model and the adjoint and in-span states.
- Stage A: the fresh adversarial search against the unchanged decomposition's 4.24.
- Stage H: step 100, which ran only if a split passed at step 10.
- The cluster check that swaps Γ's inputs between the two models, from the diagnosis, and Fable's layer-cut version of it.
- Any version of the pass test with Γ fixed to the original model's activations.

## Provenance

- Results: `experiments/qwen_maps/results/{C,S,D,K,score,stopped}.json`, `run.log`; first pod's stage C in
  `C_2026-10-28_volume.json`, its stop in `stopped_2026-10-28_S.json`.
- Run: commit 81ec101, one A100 SXM on RunPod, about 1.6 hours to the stop at stage K, $3.66 by that commit's
  record; `state.json`'s spend clock reads $6.14. GPU peak 66 GB at eval batch 2.
- Figures: `experiments/qwen_maps/plot.py`, reading the JSONs only.

## Amendment 1: do Γ's masks reproduce the update where the update is made?

*Run 2026-10-29, cluster job 44065, one L40, $0. Report: `reports/analyses/2026-10-29_analysis_e44-amend1-layer-cut.md`.
Predictions: `experiments/qwen_maps/results/predictions_amend1.json`. Results: `experiments/qwen_maps/results/amend1/`.*

**The answer.** The masks follow the fine-tuned model's output, not the layers where the update acts. The fine-tuned
weights of the first sixteen layers make 93% of the update's effect on the output. Yet Γ, given those same layers'
fine-tuned activations, produces masks that recover none of it. The masks reproduce the update only once Γ sees the
fine-tuned model's last few layers. So do-nothing's 97% on the pass test comes from Γ copying the fine-tuned output.

**What we tested.** Above, do-nothing kept almost all of the update because Γ reads the fine-tuned model's
activations. That could mean two things. Γ might read the change: the early layers' activations move, and Γ switches
on different subcomponents that carry the new behaviour. Or Γ might copy the output: it reads all 29 residual-stream
taps at once, and the last ones nearly fix the next token. To tell them apart we spliced Γ's inputs at a layer ℓ. Γ
got the fine-tuned model's residual taps and per-site readouts before ℓ, and the original model's after. The masked
model always ran the original subcomponents with Δ off, and we read its KL from the fine-tuned model on 16 maths rows.

Two shares are plotted, each 0 at the original model and 1 at the fine-tuned one.

- $m(\ell)$, red: how far the masks from cut ℓ move the masked model toward the fine-tuned one, as a share of the
  full do-nothing reproduction.
- $u(\ell)$, blue: how much of the update's own output effect the unmasked model makes with fine-tuned weights only
  in layers before ℓ. This is the comparator: if Γ reads the change, red should keep pace with blue.

We expected copying, with low confidence, and filed the rule: at layer 16, red at least half of blue reads as Γ
reading the change; less reads as copying. Three controls ran first and all held: the same read twice agreed exactly,
the splice matched the unchanged code at both ends, and the effect was present, a 1.39-nat gap.

![m and u against the cut layer](../experiments/qwen_maps/figures/amend1_layer_cut.png)

**Reading it.** Γ copies the output. The blue line climbs to 0.60 at layer 8 and 0.93 at layer 16. The red line stays
at zero through layer 16, far under the 0.46 the rule asks for. It reaches 0.31 at layer 24 and 1 only at layer 28,
although the last four layers' weights make under 1% of the effect. The diamonds split the carrier at layer 28: the
residual taps alone reach 0.99, the per-site readouts alone 0.22. The caveat is that a spliced input is a mix Γ never
saw, so a Γ that reads the change only from consistent inputs would also look like this. "Copying" is an upper bound.

| cut layer | masked KL, nats | red: masks' share | unmasked KL, nats | blue: weights' share |
|---|---|---|---|---|
| 0 | 1.496 | 0 | 1.817 | 0 |
| 8 | 1.495 | 0.0001 | 0.734 | 0.60 |
| 16 | 1.499 | −0.0025 | 0.134 | 0.93 |
| 24 | 1.069 | 0.31 | 0.011 | 0.99 |
| 28 | 0.102 | 1 | 0 | 1 |

The score flagged "non-monotone" because red at layer 8 sits 0.003 above red at layer 16. The filed rule reports the
flag without scoring it, and reads a negative share at layer 16 as copying; both values are zero at this scale.

**What it means.** The pass test as built rewards any state whose last layers' activations match the fine-tuned
model's. For the user's choice of pass test, Fable's option 1 computes every state's masks from the original model's
activations. Its cost was giving up Γ switching on different subcomponents for the fine-tuned model. On this update
that switching is the copying seen here, so option 1 gives up little that the current test measured. The 67M showed a
related failure in E49: the adversarial check could not fail plain SGD's do-nothing on new rows, so SGD's passes there
carried no weight. This is one update at one step on 16 rows of 256 tokens, with no random-update comparator on Qwen.

**Not run.** A random update on Qwen; steps 1 and 100; web rows; any split state; cuts between layers 24 and 28;
Amendment 2, the pass test with masks from the original model's activations.